<a href="https://colab.research.google.com/github/Hassnkhan0707/city-temperature-agent/blob/main/Temprature_AiAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai
print("Install done")

Install done


In [2]:
import os, time, requests
import pandas as pd
from getpass import getpass
from google import genai
from google.genai import types

os.environ["GEMINI_API_KEY"] = getpass("Paste key then press Enter: ")
print("Key saved, length:", len(os.environ["GEMINI_API_KEY"]))

Paste key then press Enter: ··········
Key saved, length: 53


In [3]:
client = genai.Client(http_options=types.HttpOptions(timeout=30000))  # 30-second limit

CANDIDATES = ["gemini-3.5-flash", "gemini-3-flash-preview"]
MODEL = None
for m in CANDIDATES:
    print("Trying", m, "...")
    try:
        r = client.models.generate_content(model=m, contents="Say hi in 3 words.")
        print("Reply:", r.text)
        MODEL = m
        break
    except Exception as e:
        print(f"{m} failed: {str(e)[:200]}")

print("Using model:", MODEL)

Trying gemini-3.5-flash ...
gemini-3.5-flash failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Trying gemini-3-flash-preview ...
Reply: Hello there, friend.
Using model: gemini-3-flash-preview


In [4]:
def get_temperature(city: str) -> dict:
    """Get the current temperature of a city in Celsius."""
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1}, timeout=10).json()
        if "results" not in geo:
            return {"error": f"City '{city}' not found"}
        p = geo["results"][0]
        w = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={"latitude": p["latitude"], "longitude": p["longitude"],
                    "current": "temperature_2m"}, timeout=10).json()
        return {"city": p["name"], "country": p.get("country", ""),
                "temperature_c": w["current"]["temperature_2m"]}
    except Exception as e:
        return {"error": str(e)}

print(get_temperature("Peshawar"))

{'city': 'Peshawar', 'country': 'Pakistan', 'temperature_c': 25.1}


In [5]:
weather_decl = types.FunctionDeclaration(
    name="get_temperature",
    description="Gets the current temperature in Celsius for a city.",
    parameters={
        "type": "object",
        "properties": {"city": {"type": "string",
                               "description": "City name, for example Lahore"}},
        "required": ["city"],
    },
)

TOOLS = {"get_temperature": get_temperature}

config = types.GenerateContentConfig(
    tools=[types.Tool(function_declarations=[weather_decl])],
    system_instruction=(
        "You are a weather assistant. For any question about the temperature "
        "of a city, call the tool. Never guess a temperature. "
        "If the question is not about temperature, answer briefly without the tool."),
)

def ask_model(contents, retries=4):
    for i in range(retries):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            msg = str(e)
            if any(code in msg for code in ["429", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE"]):
                print(f"Model busy, waiting 15 seconds... (try {i+1}/{retries})")
                time.sleep(15)
            else:
                raise
    raise RuntimeError("Model unavailable after retries")

def run_agent(question, max_steps=5, verbose=True):
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=question)])]
    tool_calls = []

    for step in range(max_steps):
        resp = ask_model(contents)
        msg = resp.candidates[0].content
        contents.append(msg)                          # memory: keep the model's turn

        calls = [p.function_call for p in msg.parts if p.function_call]
        if not calls:                                 # no tool needed -> final answer
            if verbose: print(f"[Step {step+1}] FINAL ANSWER")
            return resp.text, tool_calls

        results = []
        for fc in calls:                              # model asked for a tool
            args = dict(fc.args)
            if verbose: print(f"[Step {step+1}] THINK -> ACT : {fc.name}({args})")
            out = TOOLS[fc.name](**args)              # run the real tool
            if verbose: print(f"[Step {step+1}] OBSERVE      : {out}")
            tool_calls.append((fc.name, args))
            results.append(types.Part.from_function_response(
                name=fc.name, response={"result": out}))
        contents.append(types.Content(role="user", parts=results))   # send result back

    return "Stopped: too many steps.", tool_calls

answer, calls = run_agent("What is the temperature in Peshawar right now?")
print("\nANSWER:", answer)

[Step 1] THINK -> ACT : get_temperature({'city': 'Peshawar'})
[Step 1] OBSERVE      : {'city': 'Peshawar', 'country': 'Pakistan', 'temperature_c': 25.1}
[Step 2] FINAL ANSWER

ANSWER: The temperature in Peshawar right now is 25.1°C.


In [6]:
tests = [
    "What's the temperature in Lahore?",
    "Which is hotter right now, Karachi or Islamabad?",   # needs 2 tool calls
    "What is the capital of France?",                      # should NOT use the tool
    "Temperature in Atlantisville123?",                    # unknown city
]
for q in tests:
    print("\nQ:", q)
    a, c = run_agent(q)
    print("A:", a)
    time.sleep(3)    # small pause for free-tier limits


Q: What's the temperature in Lahore?
[Step 1] THINK -> ACT : get_temperature({'city': 'Lahore'})
[Step 1] OBSERVE      : {'city': 'Lahore', 'country': 'Pakistan', 'temperature_c': 29.5}
[Step 2] FINAL ANSWER
A: The temperature in Lahore is 29.5°C.

Q: Which is hotter right now, Karachi or Islamabad?
Model busy, waiting 15 seconds... (try 1/4)
[Step 1] THINK -> ACT : get_temperature({'city': 'Karachi'})
[Step 1] OBSERVE      : {'city': 'Karachi', 'country': 'Pakistan', 'temperature_c': 36.7}
[Step 1] THINK -> ACT : get_temperature({'city': 'Islamabad'})
[Step 1] OBSERVE      : {'city': 'Islamabad', 'country': 'Pakistan', 'temperature_c': 23.3}
[Step 2] FINAL ANSWER
A: Karachi is currently hotter than Islamabad. Karachi is 36.7°C, while Islamabad is 23.3°C.

Q: What is the capital of France?
[Step 1] FINAL ANSWER
A: The capital of France is Paris.

Q: Temperature in Atlantisville123?
[Step 1] THINK -> ACT : get_temperature({'city': 'Atlantisville123'})
[Step 1] OBSERVE      : {'error': 